# Việc C1 — Bật Mondrian theo quãng đường

> **Mentor tuần 4:** *"khoảng dự đoán hiện tại đang chia độ rộng khá đều cho phía giá thấp và phía
> giá cao. Tuy nhiên, kết quả cho thấy model bỏ sót giá cao nhiều hơn."*

Notebook này chốt lại phương án hiệu chỉnh cho khoảng tin cậy. Đây là **việc rẻ nhất của cả tuần**:
không train lại model, chỉ đổi cách tính hệ số `q`, và đã có đủ bằng chứng từ tuần 4.

## Vấn đề

Conformal toàn cục cấp **một** hệ số `q` cho mọi chuyến. Nhóm nào sai số lớn hơn mức trung bình thì
bị khoảng quá hẹp, nhóm nào sai số nhỏ hơn thì được khoảng quá rộng. Kết quả là coverage lệch nhau
tới hơn 12 điểm giữa các nhóm quãng đường — trong khi con số trung bình vẫn đẹp 89,81%.

## Mondrian

Chia dữ liệu calibration thành các nhóm, mỗi nhóm lấy phân vị riêng:

$$q_g = \text{Quantile}_{0{,}90}\big(\{res_i\}_{i \in \text{calib}, \, g(i) = g}\big)$$

Nhóm phải xác định được **tại thời điểm dự báo**. Quãng đường thoả điều kiện đó; giá thật thì không.

## Notebook này so bốn phương án

| Phương án | Nhóm |
|---|---|
| Toàn cục | không chia |
| Mondrian theo band giá dự đoán | 6 band |
| **Mondrian theo quãng đường** | 6 nhóm km |
| Mondrian theo quãng đường × band | tổ hợp |

In [ ]:
import warnings, time, sys, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

BLUE, ORANGE, GREEN, RED, PURPLE, MUT = ("#0072B2", "#E69F00", "#009E73",
                                         "#D55E00", "#CC79A7", "#666666")
INK = "#222222"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .25,
    "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10.5,
})
EVAL = Path("../model/evaluation")
DATA = Path("../data/hcm_train_ready.parquet")
HINH = Path("../docs/hinh_anh"); HINH.mkdir(parents=True, exist_ok=True)
KQ   = Path("ket_qua"); KQ.mkdir(exist_ok=True)

In [ ]:
MUC = 0.90                       # muc tin cay danh muc
CAT_GIA = [0, 50e3, 100e3, 150e3, 200e3, 300e3, np.inf]
TEN_GIA = ["<50k", "50–100k", "100–150k", "150–200k", "200–300k", ">300k"]
CAT_KM  = [0, 2, 5, 8, 12, 15, np.inf]
TEN_KM  = ["<2", "2–5", "5–8", "8–12", "12–15", ">15"]

uc = pd.read_parquet(EVAL / "uq_pred_calibration.parquet")
ut = pd.read_parquet(EVAL / "uq_pred_test.parquet")
uc = uc[uc.requested_lag_minutes == 5].copy()
te = ut[ut.requested_lag_minutes == 5].copy()

for d in (uc, te):
    # res chia cho GIA DU DOAN — luc du bao chua biet gia that
    d["res"]  = (d.hybrid_pred - d.gia_that).abs() / d.hybrid_pred
    d["band"] = pd.cut(d.hybrid_pred, CAT_GIA, labels=TEN_GIA)
    d["kmb"]  = pd.cut(d.quote_distance, CAT_KM, labels=TEN_KM)

print(f"Calibration {len(uc):,} · Test {len(te):,} (độ trễ 5 phút)")
print(f"MAPE trên test: {((te.hybrid_pred-te.gia_that).abs()/te.gia_that).mean():.2%}")

## 1. Bốn cách hiệu chỉnh

In [ ]:
N_TOI_THIEU = 200        # nhom it hon ngan nay thi dung q toan cuc cho on dinh

q_chung = uc.res.quantile(MUC)
print(f"q toàn cục = ±{q_chung:.2%}")

def q_theo_nhom(cot):
    """Phan vi rieng tung nhom, nhom qua nho thi lay q toan cuc."""
    g = uc.groupby(cot, observed=True).res.agg(["quantile", "size"])
    g.columns = ["q", "n"]
    g["q"] = np.where(g.n >= N_TOI_THIEU, uc.groupby(cot, observed=True).res.quantile(MUC), q_chung)
    return g

PA = {}
PA["Toàn cục"] = np.full(len(te), q_chung)

for ten, cot in [("Mondrian · band giá", "band"), ("Mondrian · quãng đường", "kmb")]:
    g = q_theo_nhom(cot)
    PA[ten] = te[cot].map(g.q).astype(float).fillna(q_chung).values
    print(f"\n{ten}:")
    print(g.assign(q=lambda x: (x.q*100).round(2)).to_string())

# to hop km x band
uc["o"] = uc.kmb.astype(str) + " · " + uc.band.astype(str)
te["o"] = te.kmb.astype(str) + " · " + te.band.astype(str)
g2 = uc.groupby("o", observed=True).res.agg(["quantile", "size"])
g2.columns = ["q", "n"]
g2["q"] = np.where(g2.n >= N_TOI_THIEU,
                   uc.groupby("o", observed=True).res.quantile(MUC), q_chung)
PA["Mondrian · km × band"] = te.o.map(g2.q).astype(float).fillna(q_chung).values
print(f"\nTổ hợp km × band: {len(g2)} ô, "
      f"{int((g2.n >= N_TOI_THIEU).sum())} ô đủ mẫu để có q riêng")

## 2. So bốn phương án

Ba con số cần nhìn cùng nhau: **coverage tổng** (có giữ lời hứa không), **độ rộng** (cái giá phải
trả), và **lệch coverage giữa các nhóm** (có nhóm nào bị bỏ rơi không).

In [ ]:
def danh_gia(q):
    lo, hi = te.hybrid_pred*(1-q), te.hybrid_pred*(1+q)
    trong = ((te.gia_that >= lo) & (te.gia_that <= hi))
    r = {"Coverage": trong.mean(), "Độ rộng TB": (hi-lo).mean()}
    for cot, nhan in [("kmb", "km"), ("band", "band")]:
        cov = te.assign(t=trong).groupby(cot, observed=True).t.mean()
        r[f"Lệch tối đa ({nhan})"] = float((cov - MUC).abs().max()) * 100
    r["Vượt cận trên"] = (te.gia_that > hi).mean()
    r["Dưới cận dưới"] = (te.gia_that < lo).mean()
    return r

SS = pd.DataFrame({k: danh_gia(v) for k, v in PA.items()}).T
SS["So độ rộng gốc"] = SS["Độ rộng TB"] / SS.loc["Toàn cục", "Độ rộng TB"] - 1
SS.style.format({"Coverage": "{:.2%}", "Độ rộng TB": "{:,.0f}đ",
                 "Lệch tối đa (km)": "{:.2f} điểm", "Lệch tối đa (band)": "{:.2f} điểm",
                 "Vượt cận trên": "{:.2%}", "Dưới cận dưới": "{:.2%}",
                 "So độ rộng gốc": "{:+.2%}"})

## 3. Coverage từng nhóm, trước và sau

Đây là hình cho thấy Mondrian làm gì: không kéo coverage trung bình lên, mà kéo các nhóm **về gần
nhau**.

In [ ]:
# ═════════ HÌNH MQ1 — coverage tung nhom, bon phuong an ═════════
fig, ax = plt.subplots(1, 2, figsize=(15, 5.2))
MAU_PA = {"Toàn cục": RED, "Mondrian · band giá": PURPLE,
          "Mondrian · quãng đường": GREEN, "Mondrian · km × band": BLUE}

for a, (cot, ten, nhan) in zip(ax, [("kmb", "quãng đường", TEN_KM),
                                    ("band", "band giá dự đoán", TEN_GIA)]):
    for pa, q in PA.items():
        lo, hi = te.hybrid_pred*(1-q), te.hybrid_pred*(1+q)
        trong = (te.gia_that >= lo) & (te.gia_that <= hi)
        cov = te.assign(t=trong).groupby(cot, observed=True).t.mean().reindex(nhan)
        a.plot(range(len(nhan)), cov.values*100, "o-", color=MAU_PA[pa], lw=2.2, ms=7,
               label=pa, alpha=.9 if "quãng đường" in pa else .65)
    a.axhline(MUC*100, color=INK, ls="--", lw=1.6)
    a.annotate(f"đã hứa {MUC:.0%}", (len(nhan)-1, MUC*100), fontsize=9.5, color=INK,
               ha="right", va="bottom")
    a.set_xticks(range(len(nhan))); a.set_xticklabels(nhan)
    a.set_xlabel(f"Nhóm theo {ten}"); a.set_ylabel("Coverage (%)")
    a.set_title(f"Theo {ten}", fontweight="bold")
    a.legend(frameon=False, fontsize=9)

fig.suptitle("MQ1 — Mondrian không nâng coverage trung bình, nó kéo các nhóm về gần lời hứa\n"
             "Đường đỏ là hiệu chỉnh toàn cục: nhóm chuyến dài và nhóm giá cao bị tụt rõ",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
fig.savefig(HINH / "MQ1_coverage_tung_nhom.png")
plt.show()

In [ ]:
# ═════════ HÌNH MQ2 — danh doi: le coverage vs do rong ═════════
fig, ax = plt.subplots(figsize=(9, 5.4))
for pa in PA:
    r = SS.loc[pa]
    ax.scatter(r["So độ rộng gốc"]*100, r["Lệch tối đa (km)"], s=200,
               color=MAU_PA[pa], zorder=4, label=pa)
    ax.annotate(pa, (r["So độ rộng gốc"]*100, r["Lệch tối đa (km)"]),
                fontsize=9.5, xytext=(8, 6), textcoords="offset points")
ax.axhline(0, color=MUT, lw=1); ax.axvline(0, color=MUT, lw=1)
ax.set_xlabel("Độ rộng khoảng so với toàn cục (%) — càng phải càng tốn")
ax.set_ylabel("Lệch coverage lớn nhất giữa các nhóm km (điểm) — càng thấp càng công bằng")
ax.set_title("MQ2 — Đánh đổi giữa độ rộng và mức công bằng giữa các nhóm\n"
             "Góc dưới-trái là tốt: đều hơn mà không rộng thêm",
             fontweight="bold", fontsize=12.5)
fig.tight_layout()
fig.savefig(HINH / "MQ2_danh_doi_mondrian.png")
plt.show()

## 4. Hai nhóm bị phục vụ tệ nhất

Nhóm chuyến dài và nhóm giá cao — đúng hai nhóm mà cả tuần 5 đang tập trung.

In [ ]:
hang = []
for cot, g in [("kmb", ">15"), ("kmb", "12–15"), ("band", ">300k"), ("band", "200–300k")]:
    s = te[te[cot].astype(str) == g]
    if len(s) < 30:
        continue
    r = {"Nhóm": g, "n": len(s)}
    for pa, q in PA.items():
        qq = q[te[cot].astype(str).values == g]
        lo, hi = s.hybrid_pred*(1-qq), s.hybrid_pred*(1+qq)
        r[pa] = ((s.gia_that >= lo) & (s.gia_that <= hi)).mean()
    hang.append(r)
YEU = pd.DataFrame(hang)
YEU.style.format({k: "{:.2%}" for k in PA} | {"n": "{:,}"}).hide(axis="index")

## 5. Chốt phương án và lưu

In [ ]:
# Tieu chi: le toi da theo km nho nhat, voi rang buoc do rong khong tang qua 1%
hop_le = SS[SS["So độ rộng gốc"] <= 0.01]
CHON = hop_le["Lệch tối đa (km)"].idxmin()
print(f"→ Phương án chốt: {CHON}")
print(SS.loc[CHON].to_string())

SS.to_csv(KQ / "C1_so_sanh_hieu_chinh.csv")
YEU.to_csv(KQ / "C1_nhom_yeu.csv", index=False)

q_chon = PA[CHON]
np.save(KQ / "C1_q_test.npy", q_chon)
g_km = q_theo_nhom("kmb")
g_km.to_csv(KQ / "C1_q_theo_km.csv")
json.dump({"phuong_an": CHON, "muc": MUC, "q_chung": float(q_chung),
           "coverage": float(SS.loc[CHON, "Coverage"]),
           "do_rong": float(SS.loc[CHON, "Độ rộng TB"]),
           "lech_toi_da_km": float(SS.loc[CHON, "Lệch tối đa (km)"])},
          open(KQ / "C1_ket_luan.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\nĐã lưu vào", KQ.resolve())

## 6. Kết luận cần điền

1. Phương án nào cho lệch coverage nhỏ nhất mà không tốn thêm độ rộng?
2. Nhóm `>15 km` và `>300k` cải thiện từ bao nhiêu lên bao nhiêu?
3. Chia nhóm quá mịn (tổ hợp km × band) có tốt hơn không, hay nhiều ô quá thưa khiến `q` không ổn
   định và kết quả tệ đi?

> Câu 3 đáng chú ý: chia càng mịn thì lý thuyết càng khớp, nhưng mỗi ô càng ít mẫu nên phân vị càng
> nhiễu. Đây là đánh đổi thiên lệch–phương sai quen thuộc, và số liệu sẽ cho biết điểm dừng.